# Session 1: reaction speed rises steeply with temperature
**A. Real-world problem.** Food spoils faster in a warm kitchen, and a flame needs heat to
start. A chemist measures a *rate constant* k (how fast one reaction goes) at five
temperatures. The numbers climb so steeply that a normal graph is useless. Is there a way
to draw them so that they fall on a straight line, and what does the line tell us?

**Plan.** (1) a formula for k(T), (2) a small table of "measured" data, (3) change the axes
so the data become a straight line, (4) fit that line by hand and read the activation energy.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

R_GAS = 8.314462618          # gas constant, J/(mol K)
A_TRUE = 1.0e13              # true pre-exponential factor of the toy reaction, 1/s
EA_TRUE = 150.0              # true activation energy, kJ/mol
T_DATA = np.array([600.0, 700.0, 800.0, 900.0, 1000.0])   # temperatures, K

## Step 1: the Arrhenius law
k = A * exp(-Ea / (R T)). T is the temperature in kelvin, Ea the activation energy (the
energy hill the molecules must climb), R the gas constant, A a rate scale (1/s here).
The exponent -Ea/(R T) is a pure number: Ea is converted from kJ/mol to J/mol first.

In [ ]:
def arrhenius_k(T, A=A_TRUE, Ea_kJ=EA_TRUE):
    """Rate constant (1/s) at temperature T (K)."""
    return A * np.exp(-Ea_kJ * 1000.0 / (R_GAS * np.asarray(T, dtype=float)))


def speedup(T1, T2, Ea_kJ=EA_TRUE):
    """How many times faster the reaction goes at T2 than at T1 (A cancels)."""
    return float(arrhenius_k(T2, 1.0, Ea_kJ) / arrhenius_k(T1, 1.0, Ea_kJ))

## Step 2: "measured" data
Real measurements are never exact. We make a data set from the true law and multiply every
k by a random factor of about 5 percent (a *multiplicative* error). A fixed random seed
means everybody gets the same table.

In [ ]:
def make_data(seed=1, T=T_DATA, A=A_TRUE, Ea_kJ=EA_TRUE, rel_noise=0.05):
    rng = np.random.default_rng(seed)
    k = arrhenius_k(T, A, Ea_kJ) * np.exp(rel_noise * rng.standard_normal(len(T)))
    return np.asarray(T, dtype=float), k

## Step 3: the right axes
Take logs: ln k = ln A - (Ea/R) * (1/T). Plot y = ln k against x = 1000/T and the data fall
on a straight line of slope s = -Ea/(1000 R), so Ea (kJ/mol) = -s * R with R = 8.3145.
The line is fitted by *least squares*: choose the line that makes the sum of squared
vertical gaps smallest. The hand formulas are slope = Sxy/Sxx and intercept = ybar - slope*xbar,
where Sxx = sum (x - xbar)^2 and Sxy = sum (x - xbar)(y - ybar).

In [ ]:
def to_xy(T, k):
    """Straight-line coordinates: x = 1000/T (1/K * 1000), y = ln k."""
    return 1000.0 / np.asarray(T, dtype=float), np.log(np.asarray(k, dtype=float))


def fit_line(x, y):
    """Least-squares line by hand. Returns (slope, intercept)."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    xm, ym = x.mean(), y.mean()
    slope = np.sum((x - xm) * (y - ym)) / np.sum((x - xm) ** 2)
    return float(slope), float(ym - slope * xm)


def ea_from_slope(slope):
    """Activation energy in kJ/mol from the slope of ln k against 1000/T."""
    return float(-slope * R_GAS)

## Step 4: see it
Left: k against T on a normal axis is a steep curve; the low-temperature points are squashed
against the floor. Right: ln k against 1000/T is a straight line and the fit recovers Ea.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    T, k = make_data()
    x, y = to_xy(T, k)
    s, b = fit_line(x, y)
    print("speed-up from 600 K to 700 K:", round(speedup(600, 700), 1))
    print("slope", round(s, 3), "intercept", round(b, 3), "Ea", round(ea_from_slope(s), 1), "kJ/mol")

    Tg = np.linspace(550, 1050, 200)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
    ax[0].plot(Tg, arrhenius_k(Tg), color="gray", label="true law")
    ax[0].plot(T, k, "o", label="measured")
    ax[0].set_xlabel("temperature T (K)")
    ax[0].set_ylabel("rate constant k (1/s)")
    ax[0].set_title("Normal axes: a steep curve")
    ax[0].grid(alpha=0.3)
    ax[0].legend()
    xg = np.linspace(x.min() - 0.05, x.max() + 0.05, 50)
    ax[1].plot(xg, s * xg + b, color="gray", label=f"fit, Ea = {ea_from_slope(s):.0f} kJ/mol")
    ax[1].plot(x, y, "o", label="measured")
    ax[1].set_xlabel("1000 / T (1/K x 1000)")
    ax[1].set_ylabel("ln k (k in 1/s)")
    ax[1].set_title("Arrhenius axes: a straight line")
    ax[1].grid(alpha=0.3)
    ax[1].legend()
    fig.tight_layout()
    fig.savefig(OUT / "arrhenius_axes.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Print k for Ea = 100 and Ea = 200 kJ/mol at 500, 800 and 1100 K and predict which one rises faster before you run it.
2. **Stretch.** Change the noise in `make_data` to 0.15 and then to 0.01 and compare the fitted Ea each time; explain what you see.
3. **Explain.** Explain, using the Arrhenius law, why the speed-up from 600 to 700 K (73.4) is larger than from 700 to 800 K (25.1) for the same 100 K step.

## Assessment
- **A. Conceptual.** Why does plotting ln k against 1000/T give a straight line, and what would you see if you plotted k against 1/T instead?
- **B. Numerical.** For Ea = 100 kJ/mol, by what factor does k grow from 600 K to 700 K, and what is the slope of the Arrhenius line? Compare with the 73.4 times for Ea = 150 kJ/mol.
- **C. Graph and simulation interpretation.** In the right-hand panel of the figure the fitted slope is -18.0. What activation energy does that mean, and what would a line twice as steep mean?
- **D. Design and optimisation challenge.** Design: a furnace can reach at most 600 K and you need k of at least 1000 per s with A = 1e13 per s. What is the largest activation energy that works? And what temperature does a reaction with Ea = 150 kJ/mol need to reach the same k?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    k800 = float(L.arrhenius_k(800.0))
    checks.append(("worked example: k(800 K) = 1.0e13 * exp(-150000/(8.3145*800)) = 1608 per s", abs(k800 - 1608.0) < 1.0))
    checks.append(("worked example: the exponent Ea/(R T) at 800 K is 22.55", abs(150000.0 / (L.R_GAS * 800.0) - 22.55) < 0.01))
    checks.append(("at T -> very large the law tends to A", abs(float(L.arrhenius_k(1e9)) / 1.0e13 - 1.0) < 1e-3))
    checks.append(("speed-up from 600 K to 700 K is 73.4 times", abs(L.speedup(600, 700) - 73.4) < 0.05))
    checks.append(("speed-up from 300 K to 310 K is about 7.0 times", abs(L.speedup(300, 310) - 6.96) < 0.01))
    checks.append(("speed-up from 300 K to 300 K is exactly 1", L.speedup(300, 300) == 1.0))
    # exact data lie exactly on a line whose slope gives Ea back
    x, y = L.to_xy(L.T_DATA, L.arrhenius_k(L.T_DATA))
    s, b = L.fit_line(x, y)
    checks.append(("noise-free data: slope -18.04 and intercept ln A = 29.93", abs(s + 18.04) < 0.005 and abs(b - np.log(1e13)) < 1e-6))
    checks.append(("noise-free data: Ea from the slope is 150.0 kJ/mol", abs(L.ea_from_slope(s) - 150.0) < 1e-6))
    T, k = L.make_data()
    x, y = L.to_xy(T, k)
    s, b = L.fit_line(x, y)
    checks.append(("noisy data (seed 1): fitted slope -18.005 and intercept 29.898", abs(s + 18.005) < 0.001 and abs(b - 29.898) < 0.001))
    checks.append(("noisy data (seed 1): Ea = 149.7 kJ/mol, within 1 percent of 150", abs(L.ea_from_slope(s) - 149.7) < 0.05))
    p = np.polyfit(x, y, 1)
    checks.append(("the hand formulas agree with numpy.polyfit", np.allclose([s, b], p, atol=1e-9)))
    checks.append(("two points define the line exactly", np.allclose(L.fit_line([1.0, 2.0], [3.0, 5.0]), [2.0, 1.0])))
    checks.append(("measured k rises with T (data are increasing)", bool(np.all(np.diff(k) > 0))))
    checks.append(("speed-up from 700 K to 800 K is 25.1 times (smaller than 73.4 from 600 to 700 K)", abs(L.speedup(700, 800) - 25.07) < 0.01))
    checks.append(("Ea = 100 kJ/mol: speed-up 600 to 700 K is 17.5 times and the slope is -12.03", abs(L.speedup(600, 700, 100.0) - 17.52) < 0.01 and abs(-100.0 / L.R_GAS + 12.03) < 0.005))
    checks.append(("with A = 1e13 per s, k = 1000 per s needs 783.5 K for Ea = 150 and 522.3 K for Ea = 100",
                   abs(150e3 / (L.R_GAS * np.log(1e10)) - 783.5) < 0.05 and abs(100e3 / (L.R_GAS * np.log(1e10)) - 522.3) < 0.05))
    checks.append(("at 600 K the largest Ea that still gives k = 1000 per s is 114.9 kJ/mol", abs(L.R_GAS * 600.0 * np.log(1e10) / 1000.0 - 114.9) < 0.05))
    checks.append(("app challenge: Ea = 80 kJ/mol at 500 K gives k above 1000 per s", float(L.arrhenius_k(500.0, 1e13, 80.0)) > 1000.0))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
